In [10]:
# ============================================================
# Hierarchical city POT/GPD on daily max WBT
#   log(sigma_e) = a_city + b_city * SST_basin_month
#   xi shared
#   JJAS only
#   Cities: muscat, doha, dubai, jeddah, aden
# ============================================================

import os, glob
import numpy as np
import pandas as pd
import xarray as xr
import pymc as pm
import pytensor.tensor as pt
import arviz as az
import traceback, pickle

# -----------------------
# Config
# -----------------------
NETID = "k16v981"

# --- Cities (single cell nearest) ---
CITIES = {
    "muscat":  {"lat": 23.5880, "lon": 58.3829},
    "doha":    {"lat": 25.2854, "lon": 51.5310},
    "dubai":   {"lat": 25.2048, "lon": 55.2708},
    "jeddah":  {"lat": 21.4858, "lon": 39.1925},
    "aden":    {"lat": 12.7855, "lon": 45.0187},
}
CITY_LIST = list(CITIES.keys())

# --- Variable to model from DailyPeakState files ---
TARGET_VAR = "wbt_daily_peak"   # choose from:
# "wbt_daily_peak"
# "tau_at_wbt_daily_peak"
# "t2m_at_wbt_daily_peak"
# "q_at_wbt_daily_peak"

VALID_TARGET_VARS = {
    "wbt_daily_peak": "wbt_daily_peak",
    "tau_at_wbt_daily_peak": "tau_at_wbt_daily_peak",
    "t2m_at_wbt_daily_peak": "t2m_at_wbt_daily_peak",
    "q_at_wbt_daily_peak": "q_at_wbt_daily_peak",
}

if TARGET_VAR not in VALID_TARGET_VARS:
    raise ValueError(f"TARGET_VAR must be one of {list(VALID_TARGET_VARS)}")

# --- Data paths ---
WBT_DIR  = f"/home/{NETID}/my_work/code/arabian_peninsula/bayesian_extremes/data/DailyPeakState/"
WBT_GLOB = os.path.join(WBT_DIR, "DailyPeakState-*.nc")

BASIN = "arabian_gulf"  # or gulf_oman / arabian_gulf / gulf_aden / etc.
BASIN_NC = f"/home/{NETID}/my_work/code/arabian_peninsula/bayesian_extremes/data/sst/basin_anoms/era5_sst_anom_{BASIN}_1950_2025.nc"

OUT_DIR = f"/home/{NETID}/my_work/code/arabian_peninsula/bayesian_extremes/data/wbt_sst_city_runs"
os.makedirs(OUT_DIR, exist_ok=True)
OUT_IDATA = os.path.join(
    OUT_DIR,
    f"idata_city_hier_{TARGET_VAR}_vs_sst_{BASIN}_JJAS.nc"
)

# --- Season restriction ---
MONTHS = [6,7,8,9]  # JJAS

# --- POT settings ---
Q = 0.95
MIN_EVENTS = 50  # daily scale -> use higher min than monthly

# --- GPD priors ---
XI_LOWER = -0.3
XI_UPPER = 0.5

# --- sampling ---
RANDOM_SEED = 58
DRAWS = 1500
TUNE  = 1500
CHAINS = 4
CORES  = 4
TARGET_ACCEPT = 0.98

# -----------------------
# Helpers
# -----------------------
def get_latlon_names(ds):
    lat_name = "latitude" if "latitude" in ds.coords else "lat"
    lon_name = "longitude" if "longitude" in ds.coords else "lon"
    return lat_name, lon_name

def shift_lon_180(ds, lon_name):
    lon = ds[lon_name]
    if float(lon.max()) > 180:
        lon_new = ((lon + 180) % 360) - 180
        ds = ds.assign_coords({lon_name: lon_new}).sortby(lon_name)
    return ds

def nearest_ij(lat_vals, lon_vals, lat0, lon0):
    i = int(np.argmin(np.abs(lat_vals - lat0)))
    j = int(np.argmin(np.abs(lon_vals - lon0)))
    return i, j

def pick_var(ds, target_var):
    """
    Return the requested variable from the dataset.
    Allows exact match first, then case-insensitive fallback.
    """
    if target_var in ds.data_vars:
        return target_var

    for v in ds.data_vars:
        if v.lower() == target_var.lower():
            return v

    raise KeyError(
        f"Could not find target_var='{target_var}' in dataset. "
        f"Available vars={list(ds.data_vars)}"
    )

def month_key_daily(t_daily):
    return pd.to_datetime(t_daily).to_period("M").to_timestamp()

def gpd_logp(z, sigma, xi, eps=1e-12, xi_tol=1e-6):
    sigma = sigma + eps
    t = 1 + xi * z / sigma
    logp_gpd = -pt.log(sigma) - (1 + 1/xi) * pt.log(t)
    logp_exp = -pt.log(sigma) - z / sigma
    logp = pt.switch(pt.abs(xi) < xi_tol, logp_exp, logp_gpd)
    logp = pt.switch(t > 0, logp, -np.inf)
    return pt.sum(logp)

def safe_save_idata(idata, out_path):
    try:
        az.to_netcdf(idata, out_path)
        print(f"✅ saved idata: {out_path}")
        return
    except Exception as e:
        print("⚠️ az.to_netcdf failed:", e)
        traceback.print_exc()
    try:
        pkl_path = out_path.replace(".nc",".pkl")
        with open(pkl_path,"wb") as f:
            pickle.dump(idata, f, protocol=pickle.HIGHEST_PROTOCOL)
        print(f"✅ pickled idata: {pkl_path}")
    except Exception as e:
        print("❌ pickle also failed:", e)

# -----------------------
# 1) Basin-mean SST time series (monthly)
# -----------------------
dsS = xr.open_dataset(BASIN_NC)
daS = dsS["sst_anom"]

latn, lonn = ("latitude" if "latitude" in daS.coords else "lat",
              "longitude" if "longitude" in daS.coords else "lon")
# ensure lon in [-180,180] if needed
if float(daS[lonn].max()) > 180:
    lon = daS[lonn]
    lon_new = ((lon + 180) % 360) - 180
    daS = daS.assign_coords({lonn: lon_new}).sortby(lonn)

# force monthly MS
t0 = pd.DatetimeIndex(pd.to_datetime(daS["time"].values))
is_ms = (t0.day == 1).all() and (t0.hour == 0).all()
if not is_ms:
    daS = daS.resample(time="MS").mean(skipna=True)

# basin mean (over space) -> pd.Series indexed by month-start
sst_m = daS.mean(dim=[latn, lonn], skipna=True).to_series()
sst_m.index = pd.to_datetime(sst_m.index)  # ensure datetime
dsS.close()

# also restrict to JJAS months for alignment convenience (optional)
# we'll still reindex by month_key, but this speeds sanity checks
if MONTHS is not None:
    sst_m = sst_m[sst_m.index.month.isin(MONTHS)]

# -----------------------
# 2) Load daily max WBT per city (JJAS only)
# -----------------------
files = sorted(glob.glob(WBT_GLOB))
if not files:
    raise FileNotFoundError(f"No WBT files found: {WBT_GLOB}")

def load_city_daily(city, lat0, lon0):
    ys, ts = [], []
    ij = None
    var_name = None

    for fp in files:
        ds = xr.open_dataset(fp)
        lat_name, lon_name = get_latlon_names(ds)
        ds = shift_lon_180(ds, lon_name)
        if var_name is None:
            var_name = pick_var(ds, TARGET_VAR)

        if ij is None:
            lat_vals = ds[lat_name].values
            lon_vals = ds[lon_name].values
            ij = nearest_ij(lat_vals, lon_vals, lat0, lon0)

        i, j = ij
        da = ds[var_name].isel({lat_name: i, lon_name: j})
        t = pd.to_datetime(da["time"].values)
        y = da.values.astype("float32")

        ds.close()

        ys.append(y); ts.append(t)

    y_all = np.concatenate(ys)
    t_all = pd.DatetimeIndex(np.concatenate(ts))
    o = np.argsort(t_all.values)
    t_all = t_all[o]; y_all = y_all[o]

    if MONTHS is not None:
        m = t_all.month.isin(MONTHS)
        t_all = t_all[m]; y_all = y_all[m]

    return t_all, y_all

# -----------------------
# 3) Build hierarchical event table across cities
# -----------------------
z_list, sst_list, city_id_list = [], [], []
u_by_city = {}
n_days_by_city = {}
n_exc_by_city = {}

for ci, city in enumerate(CITY_LIST):
    t, y = load_city_daily(city, CITIES[city]["lat"], CITIES[city]["lon"])
    n_days_by_city[city] = int(len(y))

    # threshold per city
    u = float(np.nanquantile(y, Q))
    exc = y > u
    z = (y[exc] - u).astype("float32")

    if z.size < MIN_EVENTS:
        raise RuntimeError(f"{city}: too few exceedances ({z.size} < {MIN_EVENTS}). Lower Q or MIN_EVENTS.")

    # monthly-step basin SST covariate for each *day*, then subset to exceedances
    mk = month_key_daily(t)
    sst_day = sst_m.reindex(mk).values.astype("float32")
    if np.isnan(sst_day).any():
        bad = np.isnan(sst_day)
        raise ValueError(f"{city}: missing SST for some JJAS days. First missing date={t[bad][0]}")

    sst_e = sst_day[exc]

    z_list.append(z)
    sst_list.append(sst_e)
    city_id_list.append(np.full(z.size, ci, dtype="int32"))

    u_by_city[city] = u
    n_exc_by_city[city] = int(z.size)

    print(f"✅ {city}: days={len(y)} exc={z.size} u={u:.3f}")

z_all   = np.concatenate(z_list).astype("float32")
sst_all = np.concatenate(sst_list).astype("float32")
cid_all = np.concatenate(city_id_list).astype("int32")

E = z_all.size
S = len(CITY_LIST)

print(f"\n✅ Built pooled event table: E={E} exceedances across S={S} cities")

# -----------------------
# 4) Hierarchical GPD model
# -----------------------
coords = {"event": np.arange(E), "city": CITY_LIST}

with pm.Model(coords=coords) as model:
    z_obs  = pm.ConstantData("z", z_all, dims="event")
    sst_e  = pm.ConstantData("sst", sst_all, dims="event")
    c_id   = pm.ConstantData("c_id", cid_all, dims="event")

    xi = pm.TruncatedNormal("xi", mu=0.05, sigma=0.15, lower=XI_LOWER, upper=XI_UPPER)

    # Hyperpriors (partial pooling)
    a_bar = pm.Normal("a_bar", 0.0, 1.0)
    b_bar = pm.Normal("b_bar", 0.0, 0.5)     # °C^-1 effect on log(sigma)

    a_sd  = pm.HalfNormal("a_sd", 0.8)
    b_sd  = pm.HalfNormal("b_sd", 0.3)

    a_z = pm.Normal("a_z", 0, 1, dims="city")
    b_z = pm.Normal("b_z", 0, 1, dims="city")

    a_city = pm.Deterministic("a_city", a_bar + a_sd * a_z, dims="city")
    b_city = pm.Deterministic("b_city", b_bar + b_sd * b_z, dims="city")

    log_sigma = a_city[c_id] + b_city[c_id] * sst_e
    sigma = pm.Deterministic("sigma", 1e-6 + pt.exp(log_sigma), dims="event")

    pm.DensityDist(
        "z_like",
        sigma, xi,
        logp=lambda z, sigma, xi: gpd_logp(z, sigma, xi),
        observed=z_obs,
        dims="event",
    )

    idata = pm.sample(
        draws=DRAWS, tune=TUNE,
        chains=CHAINS, cores=CORES,
        target_accept=TARGET_ACCEPT,
        random_seed=RANDOM_SEED,
    )

safe_save_idata(idata, OUT_IDATA)

# Save metadata needed for postprocessing
meta = {
    "cities": CITY_LIST,
    "u_by_city": u_by_city,
    "n_days_by_city": n_days_by_city,
    "n_exc_by_city": n_exc_by_city,
    "Q": Q,
    "months": MONTHS,
    "basin": BASIN,
}
pd.Series(meta).to_pickle(
    os.path.join(OUT_DIR, f"meta_city_hier_{TARGET_VAR}_vs_sst_{BASIN}_JJAS.pkl")
)
print("✅ saved meta pickle")


KeyError: 'date'

In [1]:
import numpy as np
import pandas as pd
import arviz as az
import os
import pickle
from collections import defaultdict

OUT_DIR = "../data/wbt_sst_city_runs"

# --------------------------------------------------
# Choose target variable
# --------------------------------------------------
TARGET_VAR = "wbt_daily_peak"   # choose from:
# "wbt_daily_peak"
# "tau_at_wbt_daily_peak"
# "t2m_at_wbt_daily_peak"
# "q_at_wbt_daily_peak"

VALID_TARGET_VARS = {
    "wbt_daily_peak",
    "tau_at_wbt_daily_peak",
    "t2m_at_wbt_daily_peak",
    "q_at_wbt_daily_peak",
}

if TARGET_VAR not in VALID_TARGET_VARS:
    raise ValueError(f"TARGET_VAR must be one of {sorted(VALID_TARGET_VARS)}")

CITY_TO_BASIN = {
    "muscat": "gulf_oman",
    "doha":   "arabian_gulf",
    "dubai":  "arabian_gulf",
    "jeddah": "red_sea",
    "aden":   "gulf_aden",
}

# Build basin -> cities
BASIN_TO_CITIES = defaultdict(list)
for city, basin in CITY_TO_BASIN.items():
    BASIN_TO_CITIES[basin].append(city)

WARMING_EXPTS = {
    "+0.5C": 0.5,
    "+1C": 1.0,
    "+1.5C": 1.5,
    "+2C": 2.0,
}
Q_LEVELS = [0.95, 0.99]

def gpd_quantile(u, sigma, xi, q, xi_tol=1e-6):
    xi = np.asarray(xi)
    sigma = np.asarray(sigma)
    out = np.empty(np.broadcast(xi, sigma).shape, dtype=float)

    near0 = np.abs(xi) < xi_tol
    out[~near0] = u + (sigma[~near0] / xi[~near0]) * ((1.0 - q) ** (-xi[~near0]) - 1.0)
    out[near0] = u + sigma[near0] * np.log(1.0 / (1.0 - q))
    return out

def stack_samples(da):
    return da.stack(sample=("chain", "draw")).values

def load_idata_any(out_dir, target_var, basin):
    nc_path = os.path.join(
        out_dir,
        f"idata_city_hier_{target_var}_vs_sst_{basin}_JJAS.nc"
    )
    pkl_path = os.path.join(
        out_dir,
        f"idata_city_hier_{target_var}_vs_sst_{basin}_JJAS.pkl"
    )

    # Try NetCDF first
    if os.path.exists(nc_path):
        try:
            idata = az.from_netcdf(nc_path, engine="netcdf4")
            print(f"✅ loaded NetCDF for basin='{basin}'")
            return idata, nc_path
        except Exception as e:
            print(f"⚠️ NetCDF unreadable for basin='{basin}': {e}")

    # Fall back to pickle
    if os.path.exists(pkl_path):
        try:
            with open(pkl_path, "rb") as f:
                idata = pickle.load(f)
            print(f"✅ loaded pickle for basin='{basin}'")
            return idata, pkl_path
        except Exception as e:
            print(f"⚠️ pickle unreadable for basin='{basin}': {e}")

    raise FileNotFoundError(
        f"No readable idata found for basin='{basin}'. Checked:\n"
        f"  {nc_path}\n"
        f"  {pkl_path}"
    )

all_rows = []

for basin, target_cities in BASIN_TO_CITIES.items():
    META_PATH = os.path.join(
        OUT_DIR,
        f"meta_city_hier_{TARGET_VAR}_vs_sst_{basin}_JJAS.pkl"
    )

    if not os.path.exists(META_PATH):
        print(f"⚠️ missing meta for basin='{basin}', target_var='{TARGET_VAR}'")
        continue

    try:
        idata, used_path = load_idata_any(OUT_DIR, TARGET_VAR, basin)
        print(f"   using: {used_path}")
    except Exception as e:
        print(f"⚠️ skipping basin='{basin}': {e}")
        continue

    meta = pd.read_pickle(META_PATH)

    # Optional safety check
    meta_target_var = meta.get("target_var", None)
    if meta_target_var is not None and meta_target_var != TARGET_VAR:
        print(f"⚠️ target_var mismatch for basin='{basin}': meta has '{meta_target_var}', expected '{TARGET_VAR}'")
        continue

    # Cities actually in this basin-run
    cities = list(meta["cities"])
    cities_keep = [c for c in cities if c in target_cities]

    if len(cities_keep) == 0:
        print(f"⚠️ basin='{basin}' run has cities={cities} but none match target_cities={target_cities}")
        continue

    u_by_city = meta["u_by_city"]
    n_days_by_city = meta["n_days_by_city"]
    n_exc_by_city = meta["n_exc_by_city"]

    post = idata.posterior
    xi = stack_samples(post["xi"])

    a_city = post["a_city"].stack(sample=("chain", "draw"))
    b_city = post["b_city"].stack(sample=("chain", "draw"))

    city_to_i = {c: i for i, c in enumerate(cities)}

    for city in cities_keep:
        i = city_to_i[city]
        u = float(u_by_city[city])

        a = a_city.isel(city=i).values
        b = b_city.isel(city=i).values
        sigma0 = np.exp(a)

        for label, dS in WARMING_EXPTS.items():
            sigma1 = sigma0 * np.exp(b * dS)

            for q in Q_LEVELS:
                x0 = gpd_quantile(u, sigma0, xi, q)
                x1 = gpd_quantile(u, sigma1, xi, q)
                dx = x1 - x0

                mean = float(dx.mean())
                lo, hi = az.hdi(dx, hdi_prob=0.94)

                all_rows.append({
                    "target_var": TARGET_VAR,
                    "basin_warmed": basin,
                    "city": city,
                    "warming": label,
                    "dS_C": dS,
                    "quantile": q,
                    "delta_mean": mean,
                    "delta_hdi_low": float(lo),
                    "delta_hdi_high": float(hi),
                    "n_days": int(n_days_by_city[city]),
                    "n_exc": int(n_exc_by_city[city]),
                    "u": u,
                    "months": "".join(str(m) for m in meta["months"]),
                })

impact_all = pd.DataFrame(all_rows)

out_all = os.path.join(
    OUT_DIR,
    f"{TARGET_VAR}_city_response_to_adjacent_basin_warming_ALLBASINS_JJAS.csv"
)
impact_all.to_csv(out_all, index=False)
print("✅ wrote merged adjacent-basin impacts:", out_all)


ERROR 1: PROJ: proj_create_from_database: Open of /home/k16v981/.conda/envs/my_env/share/proj failed


✅ loaded NetCDF for basin='gulf_oman'
   using: ../data/wbt_sst_city_runs/idata_city_hier_wbt_daily_peak_vs_sst_gulf_oman_JJAS.nc
⚠️ NetCDF unreadable for basin='arabian_gulf': [Errno -51] NetCDF: Unknown file format: '../data/wbt_sst_city_runs/idata_city_hier_wbt_daily_peak_vs_sst_arabian_gulf_JJAS.nc'
✅ loaded pickle for basin='arabian_gulf'
   using: ../data/wbt_sst_city_runs/idata_city_hier_wbt_daily_peak_vs_sst_arabian_gulf_JJAS.pkl
✅ loaded NetCDF for basin='red_sea'
   using: ../data/wbt_sst_city_runs/idata_city_hier_wbt_daily_peak_vs_sst_red_sea_JJAS.nc
✅ loaded NetCDF for basin='gulf_aden'
   using: ../data/wbt_sst_city_runs/idata_city_hier_wbt_daily_peak_vs_sst_gulf_aden_JJAS.nc
✅ wrote merged adjacent-basin impacts: ../data/wbt_sst_city_runs/wbt_daily_peak_city_response_to_adjacent_basin_warming_ALLBASINS_JJAS.csv
